In [1]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Agregacion con Pivote").getOrCreate()

In [3]:
estudiantes = spark.read.parquet("./data/")

# Pivotar Datos en DataFrames (Tablas Dinámicas)

La operación `pivot()` permite rotar los datos de una columna categórica para transformarlos en múltiples columnas individuales, lo que facilita enormemente la creación de reportes matriciales o tablas cruzadas.

---

## Cálculo del Esquema Resultante

Un aspecto técnico crítico que debemos considerar al usar esta operación es el crecimiento exponencial del esquema:

$$\text{Número de columnas nuevas} = (\text{Valores únicos de la columna pivote}) \times (\text{Número de agregaciones})$$

Por ejemplo, si pivotamos una columna con los meses del año (12 valores únicos) y aplicamos dos funciones de agregación simultáneas (como la suma y el promedio), Spark creará automáticamente $12 \times 2 = 24$ columnas nuevas en el DataFrame resultante.

---

## ⚠️ Advertencia de Rendimiento: Alta Cardinalidad

Si la columna elegida para pivotar tiene una **alta cardinalidad** (es decir, contiene demasiados valores distintos), el DataFrame final se volverá extremadamente ancho. Esto puede provocar sorpresas desagradables por las siguientes razones:

1. **Saturación de Memoria:** Gestionar un esquema con cientos o miles de columnas degrada drásticamente el rendimiento del motor Catalyst de Spark.
2. **Alto costo de Red (*Shuffle*):** Si no se especifican los valores, Spark se ve obligado a realizar un paso previo para escanear todo el conjunto de datos y descubrir cuáles son todos los valores únicos, lo que ralentiza el proceso.

### La Solución: Filtrar los valores de interés

In [4]:
estudiantes.printSchema()

root
 |-- nombre: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- peso: long (nullable = true)
 |-- graduacion: long (nullable = true)



In [5]:
# Queremos hallar el promedio de peso por año y graduacion

from pyspark.sql.functions import min, max, avg, col

In [7]:
estudiantes.groupBy("graduacion").pivot("sexo").agg(avg("peso")).show()

+----------+----+----+
|graduacion|   F|   M|
+----------+----+----+
|      2001|65.0|76.0|
|      2000|50.0|77.5|
+----------+----+----+



In [8]:
# si la columna de genero en este caso tendria 3 posibles valroes unicos habra 3 columnas en la tabla de resultado

In [9]:
estudiantes.groupBy("graduacion").pivot("sexo").agg(avg("peso"), min("peso"), max("peso")).show()

+----------+-----------+-----------+-----------+-----------+-----------+-----------+
|graduacion|F_avg(peso)|F_min(peso)|F_max(peso)|M_avg(peso)|M_min(peso)|M_max(peso)|
+----------+-----------+-----------+-----------+-----------+-----------+-----------+
|      2001|       65.0|         65|         65|       76.0|         76|         76|
|      2000|       50.0|         50|         50|       77.5|         75|         80|
+----------+-----------+-----------+-----------+-----------+-----------+-----------+



In [12]:
# Pivot tiene un parametro para poder pasarle una lista para una columna especifica

estudiantes.groupBy("graduacion").pivot("sexo", ["M"]).agg(avg("peso"), min("peso"), max("peso")).show()

+----------+-----------+-----------+-----------+
|graduacion|M_avg(peso)|M_min(peso)|M_max(peso)|
+----------+-----------+-----------+-----------+
|      2001|       76.0|         76|         76|
|      2000|       77.5|         75|         80|
+----------+-----------+-----------+-----------+



In [13]:
# Pivot tiene un parametro para poder pasarle una lista para una columna especifica

estudiantes.groupBy("graduacion").pivot("sexo", ["F"]).agg(avg("peso"), min("peso"), max("peso")).show()

+----------+-----------+-----------+-----------+
|graduacion|F_avg(peso)|F_min(peso)|F_max(peso)|
+----------+-----------+-----------+-----------+
|      2001|       65.0|         65|         65|
|      2000|       50.0|         50|         50|
+----------+-----------+-----------+-----------+

